In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

True

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b")

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [3]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\utyag\AppData\Local\Temp\ipykernel_6764\2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
config={"configurable":{"session_id":"chat1"}}

In [5]:
from langchain_core.messages import HumanMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Ujjwal and I am a Senior AI/ML Engineer")],
    config=config
)

In [6]:
print(response.content.split("</think>")[-1])



Hi Ujjwal, nice to meet you! It's great to connect with a Senior AI/ML Engineer. Whether you're tackling model architecture, MLOps pipelines, scaling inference, experimental design, code review, or evaluating the latest research, I'm here to help. 

What are you currently working on, or how can I assist you today?


In [7]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
print(response.content.split("</think>")[-1])




Your name is Ujjwal. Let me know how I can assist you with your AI/ML work today!


In [8]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1,
)
print(response.content.split("</think>")[-1])




I don't actually know your name! I don't have access to personal information unless you've shared it with me in this conversation. If you'd like to tell me your name, I'd be happy to use it going forward. 😊


In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
print(response.content.split("</think>")[-1])



Nice to meet you, John! I've got that noted for our conversation. How can I help you today? 😊


In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
print(response.content.split("</think>")[-1])



Your name is John! How can I help you today? 😊


### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [11]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [13]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Ujjwal")]})

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Ujjwal"\n   - This is a simple greeting and introduction.\n   - No specific question or request is made.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name: "Ujjwal"\n   - Intent: Introduction/Starting a conversation\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting\n   - Use the user\'s name (Ujjwal) to personalize the response\n   - Offer assistance\n   - Keep it friendly and open-ended to encourage further interaction\n\n4.  **Draft Response (Mental):**\n   Hi Ujjwal! Nice to meet you. How can I assist you today? Whether you have questions, need help with something, or just want to chat, I\'m here for you!\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional, welcoming\n   - Check accuracy: Uses correct name, offers help\n   - Keep it concise\n   - Matches the user\'s casual tone while remaining helpful\n\n   The

In [14]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Krish"\n   - This is a simple greeting and introduction.\n   - No specific question or request is made.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name provided: "Krish"\n   - Tone: Friendly, casual\n\n3.  **Determine Appropriate Response:**\n   - Acknowledge the greeting\n   - Use the provided name\n   - Maintain a friendly, helpful tone\n   - Offer assistance\n   - Keep it concise\n\n4.  **Draft Response (Mental):**\n   Hi Krish! Nice to meet you. How can I help you today?\n\n5.  **Refine Response:**\n   - Check for correctness: Name spelled correctly? Yes.\n   - Tone appropriate? Yes.\n   - Open-ended to encourage next interaction? Yes.\n   - Matches system prompt? Yes, helpful and direct.\n\n   Final: "Hi Krish! Nice to meet you. How can I help you today?"\n\n6.  **Output Generation:** (matches the refined response)✅\n</think>\n\nHi Krish! Nice 

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

print(response.content.split("</think>")[-1])



Your name is Krish! How can I help you today? 😊


In [18]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
print(response.content.split("</think>")[-1])



नमस्ते क्रिष! आपका स्वागत है। बताइए, आज मैं आपकी कैसे सहायता कर सकता हूँ? 😊


Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config = {"configurable": {"session_id": "chat5"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Krish"
   - Language: English
   - Context: Greeting and self-introduction
   - My role: Helpful assistant, must respond in Hindi as per system instruction

2.  **Identify Key Requirements:**
   - Acknowledge the greeting
   - Acknowledge the name (Krish)
   - Respond in Hindi (as per instruction: "Answer all questions to the best of your ability in Hindi.")
   - Keep it friendly and helpful

3.  **Formulate Response (Mental Draft in Hindi):**
   - नमस्ते क्रिष! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता हूँ?
   - (Namaste Krish! Welcome. How can I help you?)

4.  **Check Against Constraints:**
   - Is it in Hindi? Yes.
   - Does it acknowledge the name? Yes.
   - Is it helpful and polite? Yes.
   - Matches system instruction? Yes.

5.  **Refine Response (if needed):**
   - "नमस्ते क्रिष! आपका स्वागत है। बताइये, आज मैं आपकी कैसे सहायता कर सकता हूँ?"
   - This is natural, polite, and fully in 

In [22]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [23]:
response.content.split("</think>")[-1]

'\n\nआपका नाम **क्रिष** है। आपने ही पहली बार में अपना परिचय दिया था। बताइए, आज मैं आपकी और किस काम में सहायता कर सकता हूँ? 😊'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [24]:
from langchain_core.messages import SystemMessage, AIMessage, trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [25]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content.split("</think>")[-1]

"\n\nI don't actually know! I don't have memory of past conversations or access to personal information about you. But if you tell me your favorite, I'd love to hear it—or I can help you brainstorm new flavors to try! 🍦✨"

In [26]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content.split("</think>")[-1]

'\n\nYou asked "whats 2 + 2".'

In [27]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content.split("</think>")[-1]

"\n\nI don't actually know! You haven't mentioned it in our conversation. What should I call you? 😊"

In [29]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content.split("</think>")[-1]

"\n\nYou haven't asked a math problem yet in this conversation—this is actually our first message! Please feel free to share the problem you'd like help with, and I'll gladly walk you through the solution."